In [1]:
dataset = "pv-italy"

DATASET_CFG = {
    "lightsource": {
        "altieri_perm": ["timestamp", "node", "power", "temperature", "irradiance", "pressure", "windspeed", "windbearing", "humidity", "dewpoint", "cloudcover", "altitude", "azimuth"],
        "node_col": "node",
        "timestamp_col": "timestamp",
    },
    "pv-italy": {
        "node_col": "idsito",
        "timestamp_col": "data",
        "altieri_perm": ["data", "idsito", "kwh", "temperatura_ambiente", "irradiamento", "pressure", "windspeed", "windbearing", "humidity", "dewpoint", "cloudcover", "icon", "altitude", "azimuth" ],
    }
}

dataset_cfg = DATASET_CFG[dataset]

# Dataset

In [2]:
import pandas as pd
import numpy as np

In [3]:
data_path = f"../dvc-stages-outputs/{dataset}/raw/{dataset}.parquet"
df = pd.read_parquet(data_path)

In [4]:
df.sort_values(by=[dataset_cfg["node_col"], dataset_cfg["timestamp_col"]], inplace=True)
df = df[dataset_cfg["altieri_perm"]]
df

,data,idsito,kwh,temperatura_ambiente,irradiamento,pressure,windspeed,windbearing,humidity,dewpoint,cloudcover,icon,altitude,azimuth
0,2012-01-02 02:00:00,0.000,0.000,0.212,0.000,0.000,0.054,0.557,0.82,0.436,0.00,0.083,0.117,0.147
1,2012-01-02 03:00:00,0.000,0.000,0.212,0.000,0.762,0.020,0.441,0.85,0.434,0.11,0.083,0.131,0.207
2,2012-01-02 04:00:00,0.000,0.000,0.212,0.000,0.762,0.086,0.450,0.84,0.431,0.11,0.083,0.175,0.270
3,2012-01-02 05:00:00,0.000,0.000,0.212,0.000,0.762,0.086,0.471,0.70,0.440,0.11,0.083,0.231,0.308
4,2012-01-02 06:00:00,0.000,0.010,0.212,0.002,0.762,0.086,0.493,0.82,0.430,0.11,0.083,0.290,0.335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276483,2014-05-06 16:00:00,0.425,0.025,0.500,0.054,0.758,0.036,0.163,0.56,0.598,0.31,0.000,0.502,0.682
276484,2014-05-06 17:00:00,0.425,0.003,0.485,0.010,0.758,0.046,0.133,0.58,0.596,0.31,0.000,0.737,0.721
276485,2014-05-06 18:00:00,0.425,0.000,0.470,0.000,0.758,0.021,0.191,0.63,0.589,0.31,0.083,0.678,0.749
276486,2014-05-06 19:00:00,0.425,0.000,0.455,0.000,0.758,0.023,0.283,0.73,0.588,0.31,0.083,0.619,0.773


# Checking

In [5]:
node = 2
# 1-7, 
timestamp = 18

In [6]:
grouped = df.groupby(dataset_cfg["node_col"])

tensor = []
grouped = sorted(grouped, key=lambda x: x[0])
for name, group in grouped:
    tensor.append(group.drop(columns=[dataset_cfg["node_col"], dataset_cfg["timestamp_col"]]).to_numpy())

np_tensor = np.stack(tensor, axis=1)
print(f"Reverse Engineered Shape of the altieri tensor: {np_tensor.shape}")
print(f"Features in the first row: {np_tensor[timestamp, node]}")

Reverse Engineered Shape of the altieri tensor: (16264, 17, 12)
Features in the first row: [0.    0.333 0.    0.76  0.041 0.503 0.86  0.532 0.31  0.75  0.43  0.721]


In [7]:
alt_tensor = np.load(f"../dvc-stages-outputs/{dataset}/{dataset}.npz")["data"]
print(f"Original Shape of the altieri tensor: {alt_tensor.shape}")
print(f"Features in the first row: {alt_tensor[timestamp, node]}")

Original Shape of the altieri tensor: (16264, 17, 12)
Features in the first row: [0.    0.318 0.    0.76  0.116 0.555 0.88  0.541 0.31  0.75  0.428 0.722]


In [8]:
RTOL = 0.00001
ATOL = 1e-8

# RTOL = 0
# ATOL = 1e-8

In [9]:
perm = []
for rev in range(np_tensor.shape[1]):
    for orig in range(alt_tensor.shape[1]):
        np_tensor_node = np_tensor[:, rev, :]
        alt_tensor_node = alt_tensor[:, orig, :]

        if np.allclose(np_tensor_node, alt_tensor_node, rtol=RTOL, atol=ATOL, equal_nan=True):
            perm.append((rev, orig))

if len(perm) != np_tensor.shape[1]:
    print(f"Warning: Not all nodes matched. Found {len(perm)} matches out of {np_tensor.shape[1]} nodes.")
else:
    alt_perm = [orig for rev, orig in sorted(perm, key=lambda x: x[0])]
    print(f"Altieri Nodes Permutation: {alt_perm}")
    alt_tensor = alt_tensor[:, alt_perm, :]

Altieri Nodes Permutation: [0, 8, 9, 10, 11, 12, 13, 14, 15, 16, 1, 2, 3, 4, 5, 6, 7]


In [10]:
are_close = np.allclose(
    np_tensor, alt_tensor, 
    rtol=RTOL, atol=ATOL, 
    equal_nan=True
)

print(f"Are the two tensors close? {are_close}")

mask = np.isclose(
    np_tensor, alt_tensor,
    rtol=RTOL, atol=ATOL,
    equal_nan=True
)
# count how many values are not close
not_close_pctg = (np.size(mask) - np.count_nonzero(mask)) / np.size(mask) * 100
print(f"Percentage of values that are not close: {not_close_pctg:.2f}%")

Are the two tensors close? True
Percentage of values that are not close: 0.00%


In [11]:
not_equal_mask = ~mask
not_equal_indexes = np.argwhere(not_equal_mask)
for idx in not_equal_indexes:
    print(f"Index: {idx}\n")
    print(f"    np_tensor value: {np_tensor[tuple(idx)]}")
    print(f"    alt_tensor value: {alt_tensor[tuple(idx)]}")
    choice = input("> ")
    if choice == "q":
        break

In [12]:
alt_tensor_rounded = np.round(alt_tensor, decimals=15)
mask = np.equal(np_tensor, alt_tensor_rounded)

print(f"Are the two tensors exactly equal? {np.all(mask)}")

not_close_pctg = (np.size(mask) - np.count_nonzero(mask)) / np.size(mask) * 100
print(f"Percentage of values that are not equal: {not_close_pctg:.2f}%")

Are the two tensors exactly equal? True
Percentage of values that are not equal: 0.00%


In [13]:
not_equal_mask = ~mask
not_equal_indexes = np.argwhere(not_equal_mask)
for idx in not_equal_indexes:
    print(f"Index: {idx}\n")
    print(f"    np_tensor value: {np_tensor[tuple(idx)]}")
    print(f"    alt_tensor value: {alt_tensor[tuple(idx)]}")
    choice = input("> ")
    if choice == "q":
        break

# Plots

In [14]:
df[dataset_cfg["node_col"]] = (df[dataset_cfg["node_col"]] * (41.0 - 1.0) + 1.0).astype(int)
df

,data,idsito,kwh,temperatura_ambiente,irradiamento,pressure,windspeed,windbearing,humidity,dewpoint,cloudcover,icon,altitude,azimuth
0,2012-01-02 02:00:00,1,0.000,0.212,0.000,0.000,0.054,0.557,0.82,0.436,0.00,0.083,0.117,0.147
1,2012-01-02 03:00:00,1,0.000,0.212,0.000,0.762,0.020,0.441,0.85,0.434,0.11,0.083,0.131,0.207
2,2012-01-02 04:00:00,1,0.000,0.212,0.000,0.762,0.086,0.450,0.84,0.431,0.11,0.083,0.175,0.270
3,2012-01-02 05:00:00,1,0.000,0.212,0.000,0.762,0.086,0.471,0.70,0.440,0.11,0.083,0.231,0.308
4,2012-01-02 06:00:00,1,0.010,0.212,0.002,0.762,0.086,0.493,0.82,0.430,0.11,0.083,0.290,0.335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276483,2014-05-06 16:00:00,18,0.025,0.500,0.054,0.758,0.036,0.163,0.56,0.598,0.31,0.000,0.502,0.682
276484,2014-05-06 17:00:00,18,0.003,0.485,0.010,0.758,0.046,0.133,0.58,0.596,0.31,0.000,0.737,0.721
276485,2014-05-06 18:00:00,18,0.000,0.470,0.000,0.758,0.021,0.191,0.63,0.589,0.31,0.083,0.678,0.749
276486,2014-05-06 19:00:00,18,0.000,0.455,0.000,0.758,0.023,0.283,0.73,0.588,0.31,0.083,0.619,0.773


In [15]:
from plots import plot_data
import os

feature = "irradiamento"
renamed_df = df.rename(
    columns={
        dataset_cfg["node_col"]: "site_id", 
        dataset_cfg["timestamp_col"]: "timestamp"
    }
)
for node in range(1, 17):
    if node == 18:
        continue
    for feature in dataset_cfg["altieri_perm"]:
        if feature in [dataset_cfg["node_col"], dataset_cfg["timestamp_col"]]:
            continue
        fig = plot_data(
            renamed_df, 
            feature, 
            site_id=node, nan_indexes=None
        )
        os.makedirs(f"./plots/{node}", exist_ok=True)
        fig.write_html(f"./plots/{node}/{feature}.html")